# 09 – Cải thiện mô hình: toàn bộ độ dài câu, thêm PhoWhisper, kết hợp và ensemble

Ba cải tiến, đều giữ mô hình nền **đóng băng** và dùng head weighted-sum + MLP như notebook 07:

| Bộ đặc trưng | Mô tả |
|---|---|
| **A. XLS-R 12 giây** | Mốc so sánh. Dùng lại cache của notebook 02, giữ 13 tầng |
| **B. XLS-R toàn câu** | Chia câu thành các cửa sổ 12 giây, lấy trung bình mọi frame của **cả câu** (giới hạn 60 giây) |
| **C. PhoWhisper-large** | Encoder Whisper đã fine-tune cho tiếng Việt (VinAI). Trong bài báo ViMD, PhoWhisper-base cho kết quả tốt nhất. Cửa sổ 30 giây |
| **D. Kết hợp B + C** | Hai nhánh weighted-sum riêng → ghép lại → MLP |

- Mỗi bộ đặc trưng train với LS và Geo-LS, mỗi cấu hình 3 seed. Sau đó thêm **ensemble** (trung bình xác suất).
- Chọn mô hình cuối **theo valid**.
- Có thêm **probe từng tầng** của cả hai mô hình nền, để biết thông tin phương ngữ nằm ở đâu.

**Thông tin chạy:**
- **Runtime:** GPU L4 (A100 nhanh hơn).
- **Thời gian:** trích XLS-R toàn câu khoảng 20–30 phút; PhoWhisper-large khoảng 45–75 phút; train head khoảng 10–15 phút.
- Đặc trưng được cache trên Drive theo từng shard. Nếu bị ngắt, chỉ cần *Run all* lại.
- **Kết quả:** `results/improve_report.json` và `fig_layers_backbones.png`. Gửi cả hai cho Claude.

In [ ]:
# ==== CẤU HÌNH ====
DRIVE_ROOT = "/content/drive/MyDrive/vimd_dl"
PROVINCE_COL = "province_code"
XLSR_MODEL = "facebook/wav2vec2-xls-r-300m"
XLSR_KEEP = 12                       # 13 trạng thái: embedding + tầng 1–12
XLSR_WIN_SEC, MAX_SEC = 12.0, 60.0
WHISPER_MODEL = "vinai/PhoWhisper-large"   # có thể đổi sang "vinai/PhoWhisper-medium" nếu muốn nhanh hơn
WHISPER_WIN_SEC = 30.0
XLSR_BATCH, WHISPER_BATCH = 16, 8
SEEDS = [0, 1, 2]
RUN_LAYER_PROBE = True

In [ ]:
import os, io, glob, json, time, warnings
import numpy as np, pandas as pd, pyarrow.parquet as pq, soundfile as sf
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import f1_score, top_k_accuracy_score
warnings.filterwarnings("ignore")
T0 = time.time()
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
LOCAL = "/content" if os.path.isdir("/content") else os.path.abspath("_work")
LOCAL_DATA, FEAT_DIR, RES_DIR = f"{LOCAL}/vimd16k", f"{DRIVE_ROOT}/features", f"{DRIVE_ROOT}/results"
os.makedirs(LOCAL_DATA, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if device.type == "cuda" else None
autocast = lambda: torch.autocast(device_type=device.type, dtype=AMP or torch.bfloat16, enabled=AMP is not None)
print("Thiết bị:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "", "| AMP:", AMP)

meta = pd.read_csv(f"{DRIVE_ROOT}/metadata.csv", low_memory=False)
meta[PROVINCE_COL] = meta[PROVINCE_COL].astype(str)
prov = pd.read_csv(f"{DRIVE_ROOT}/provinces.csv"); prov[PROVINCE_COL] = prov[PROVINCE_COL].astype(str)
CLASSES = sorted(meta[PROVINCE_COL].unique()); C = len(CLASSES)
y = meta[PROVINCE_COL].map({c: i for i, c in enumerate(CLASSES)}).values
use = ~meta["exclude"].astype(bool).values
tr, va, te = [(meta["split"].values == s) & use for s in ("train", "valid", "test")]
pidx = prov.set_index(PROVINCE_COL)
NAMES = [pidx.loc[c, "province_name"] for c in CLASSES]
region_of = np.array([pidx.loc[c, "region"] for c in CLASSES])
yt = y[te]; N = len(meta)
GROUPS = {s: (g.index.values, g["row"].values) for s, g in meta.groupby("shard")}
print(f"{C} tỉnh | train {tr.sum()} | valid {va.sum()} | test {te.sum()} | độ dài trung vị {meta['duration'].median():.1f}s, "
      f"tỉ lệ câu dài hơn 12s: {(meta['duration'] > 12).mean():.0%}")

COORD = {
 "CaoBang": (22.666, 106.258), "LangSon": (21.853, 106.761), "QuangNinh": (20.951, 107.080), "HaiPhong": (20.845, 106.688),
 "ThaiBinh": (20.446, 106.336), "NamDinh": (20.420, 106.168), "PhuTho": (21.322, 105.402), "ThaiNguyen": (21.594, 105.848),
 "YenBai": (21.705, 104.875), "TuyenQuang": (21.823, 105.214), "HaGiang": (22.823, 104.984), "LaoCai": (22.486, 103.971),
 "LaiChau": (22.396, 103.458), "SonLa": (21.326, 103.917), "DienBien": (21.386, 103.023), "HoaBinh": (20.817, 105.338),
 "HaNoi": (21.028, 105.854), "HaiDuong": (20.940, 106.333), "NinhBinh": (20.251, 105.975), "VinhPhuc": (21.310, 105.597),
 "HungYen": (20.646, 106.051), "HaNam": (20.541, 105.914), "BacKan": (22.147, 105.835), "BacGiang": (21.273, 106.195),
 "BacNinh": (21.186, 106.076), "ThanhHoa": (19.807, 105.776), "NgheAn": (18.679, 105.682), "HaTinh": (18.343, 105.906),
 "QuangBinh": (17.468, 106.622), "QuangTri": (16.816, 107.100), "ThuaThienHue": (16.464, 107.586), "DaNang": (16.054, 108.202),
 "QuangNam": (15.573, 108.474), "QuangNgai": (15.120, 108.792), "BinhDinh": (13.776, 109.224), "PhuYen": (13.096, 109.309),
 "KhanhHoa": (12.239, 109.197), "NinhThuan": (11.565, 108.988), "BinhThuan": (10.928, 108.102), "KonTum": (14.350, 108.000),
 "GiaLai": (13.983, 108.000), "DakLak": (12.667, 108.038), "DakNong": (12.004, 107.690), "LamDong": (11.940, 108.458),
 "HoChiMinh": (10.776, 106.701), "DongNai": (10.957, 106.843), "BinhDuong": (10.980, 106.651), "BaRiaVungTau": (10.496, 107.169),
 "TayNinh": (11.310, 106.098), "BinhPhuoc": (11.535, 106.883), "LongAn": (10.535, 106.413), "TienGiang": (10.360, 106.360),
 "BenTre": (10.241, 106.376), "VinhLong": (10.254, 105.972), "TraVinh": (9.935, 106.345), "CanTho": (10.045, 105.747),
 "HauGiang": (9.784, 105.470), "SocTrang": (9.603, 105.980), "BacLieu": (9.294, 105.727), "CaMau": (9.176, 105.150),
 "DongThap": (10.460, 105.633), "AnGiang": (10.386, 105.435), "KienGiang": (10.012, 105.081),
}
LAT = np.array([COORD[n][0] for n in NAMES]); LON = np.array([COORD[n][1] for n in NAMES])
p_ = np.pi / 180
DIST = 2 * 6371 * np.arcsin(np.sqrt(np.sin((LAT[None] - LAT[:, None]) * p_ / 2) ** 2 + np.cos(LAT[:, None] * p_)
                                    * np.cos(LAT[None] * p_) * np.sin((LON[None] - LON[:, None]) * p_ / 2) ** 2))
EYE = torch.eye(C, device=device); DIST_t = torch.tensor(DIST, dtype=torch.float32, device=device)

def target_matrix(kind, p):
    if kind == "ls":
        return (1 - p) * EYE + p / C
    if kind == "geols":
        eps, tau = p
        nb = torch.exp(-DIST_t / tau) * (1 - EYE)
        return (1 - eps) * EYE + eps * nb / nb.sum(1, keepdim=True)
    return EYE

def metrics(P, yy):
    pred = P.argmax(1); d = DIST[yy, pred]
    return dict(acc=(pred == yy).mean(), macro_f1=f1_score(yy, pred, average="macro"),
                top5=top_k_accuracy_score(yy, P, k=min(5, C - 1), labels=np.arange(C)),
                region_acc=(region_of[pred] == region_of[yy]).mean(), mean_err_km=d.mean(),
                within_100km=(d <= 100).mean())

rep = f"{RES_DIR}/geo_ls_report_13layers.json"
ch = json.load(open(rep if os.path.exists(rep) else f"{RES_DIR}/geo_ls_report.json"))["chosen"]
LOSSES = {"ls": ("ls", float(ch["ls"])), "geols": ("geols", tuple(float(v) for v in ch["geols"].strip("()").split(",")))}
print("Hàm mất mát:", LOSSES)

for f in sorted(glob.glob(f"{DRIVE_ROOT}/data16k/*.parquet")):
    dst = f"{LOCAL_DATA}/{os.path.basename(f)}"
    if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(f):
        import shutil; shutil.copy(f, dst)

def shard_waves(shard):
    idx, rows = GROUPS[shard]
    col = pq.read_table(f"{LOCAL_DATA}/{shard}", columns=["audio"]).column(0)
    return idx, [sf.read(io.BytesIO(col[int(r)].as_py()), dtype="float32")[0][:int(MAX_SEC * 16000)] for r in rows]

def windows(w, win):
    """Chia waveform thành các cửa sổ dài `win` mẫu; đoạn cuối < 2s được gộp vào cửa sổ trước."""
    out = [w[i:i + win] for i in range(0, len(w), win)]
    if len(out) > 1 and len(out[-1]) < 2 * 16000:
        out[-2] = np.concatenate([out[-2], out[-1]]); out.pop()
    return out

def save_np(path, arr):
    with open(path + ".part", "wb") as f:
        np.save(f, arr)
    os.replace(path + ".part", path)

def extract(name, embed_windows, win_sec, batch):
    """embed_windows(list_of_waves) -> (pooled: B×L×D float32 numpy, n_frames: B). Kết quả gộp theo câu, có trọng số theo số frame."""
    fdir = f"{FEAT_DIR}/{name}"; os.makedirs(fdir, exist_ok=True)
    todo = [s for s in GROUPS if not os.path.exists(f"{fdir}/{s}.npy")]
    print(f"[{name}] {len(GROUPS) - len(todo)}/{len(GROUPS)} shard đã có cache")
    t, done = time.time(), 0
    for k, s in enumerate(todo, 1):
        idx, waves = shard_waves(s)
        items = [(u, wv) for u, w in enumerate(waves) for wv in windows(w, int(win_sec * 16000))]
        items.sort(key=lambda it: -len(it[1]))
        acc, cnt = None, np.zeros(len(waves))
        for i in range(0, len(items), batch):
            chunk = items[i:i + batch]
            pooled, nfr = embed_windows([wv for _, wv in chunk])
            if acc is None:
                acc = np.zeros((len(waves),) + pooled.shape[1:], np.float64)
            for (u, _), pv, n in zip(chunk, pooled, nfr):
                acc[u] += pv * n; cnt[u] += n
        save_np(f"{fdir}/{s}.npy", (acc / cnt[:, None, None]).astype(np.float16)); done += len(waves)
        if k % 5 == 0 or k == len(todo):
            el = time.time() - t
            print(f"  [{name}] {k}/{len(todo)} shard | {el / 60:.1f} phút | còn ~{el / done * (N * len(todo) / len(GROUPS) - done) / 60:.0f} phút", flush=True)
    first = np.load(f"{fdir}/{next(iter(GROUPS))}.npy")
    X = np.zeros((N,) + first.shape[1:], np.float16)
    for s, (idx, _) in GROUPS.items():
        X[idx] = np.load(f"{fdir}/{s}.npy")
    return X

## 1. Trích đặc trưng B: XLS-R trên toàn bộ câu (13 tầng)

In [ ]:
from transformers import AutoModel, AutoFeatureExtractor
xname = f"full{XLSR_KEEP + 1}L_{XLSR_MODEL.split('/')[-1]}_{int(XLSR_WIN_SEC)}s"
if not all(os.path.exists(f"{FEAT_DIR}/{xname}/{s}.npy") for s in GROUPS):
    xenc = AutoModel.from_pretrained(XLSR_MODEL, num_hidden_layers=XLSR_KEEP, layerdrop=0.0)
    if getattr(xenc.config, "do_stable_layer_norm", False):
        xenc.encoder.layer_norm = nn.Identity()
    xenc = xenc.to(device).eval()
else:
    xenc = None

@torch.no_grad()
def xlsr_embed(ws):
    L = max(len(w) for w in ws)
    x = np.zeros((len(ws), L), np.float32); m = np.zeros((len(ws), L), np.int64)
    for j, w in enumerate(ws):
        w = (w - w.mean()) / (w.std() + 1e-7); x[j, :len(w)] = w; m[j, :len(w)] = 1
    x, m = torch.from_numpy(x).to(device), torch.from_numpy(m).to(device)
    with autocast():
        hs = xenc(x, attention_mask=m, output_hidden_states=True).hidden_states[:XLSR_KEEP + 1]
    fl = xenc._get_feat_extract_output_lengths(m.sum(1)).long()
    fm = (torch.arange(hs[0].shape[1], device=device)[None] < fl[:, None]).float()
    pooled = torch.stack([(h.float() * fm[..., None]).sum(1) / fm.sum(1, keepdim=True) for h in hs], 1)
    return pooled.cpu().numpy(), fl.cpu().numpy()

XB = extract(xname, xlsr_embed, XLSR_WIN_SEC, XLSR_BATCH)
del xenc; torch.cuda.empty_cache()
print("B:", XB.shape)

## 2. Trích đặc trưng C: encoder PhoWhisper (tất cả các tầng)
Whisper luôn nhận đầu vào đệm đủ 30 giây. Ở đây chỉ lấy trung bình trên các frame thật, bỏ phần đệm.
Hidden state của từng tầng được lấy bằng forward hook, nên chạy được với mọi phiên bản `transformers`.

In [ ]:
wname = f"enc_{WHISPER_MODEL.split('/')[-1]}"
if not all(os.path.exists(f"{FEAT_DIR}/{wname}/{s}.npy") for s in GROUPS):
    from transformers import WhisperModel
    wfe = AutoFeatureExtractor.from_pretrained(WHISPER_MODEL)
    wenc = WhisperModel.from_pretrained(WHISPER_MODEL, torch_dtype=torch.float16 if device.type == "cuda" else torch.float32).encoder
    wenc = wenc.to(device).eval()
    CAP = []
    wenc.layers[0].register_forward_pre_hook(lambda mod, args: CAP.append(args[0]))
    for lyr in wenc.layers:
        lyr.register_forward_hook(lambda mod, inp, out: CAP.append(out[0] if isinstance(out, tuple) else out))
else:
    wenc = None

@torch.no_grad()
def whisper_embed(ws):
    feats = wfe([w for w in ws], sampling_rate=16000, return_tensors="pt").input_features
    CAP.clear()
    wenc(feats.to(device, dtype=next(wenc.parameters()).dtype))
    hs = CAP[:len(wenc.layers) + 1]
    T = hs[0].shape[1]
    fl = torch.tensor([min(T, int(np.ceil(len(w) / 16000 * 50))) for w in ws], device=device)
    fm = (torch.arange(T, device=device)[None] < fl[:, None]).float()
    pooled = torch.stack([(h.float() * fm[..., None]).sum(1) / fm.sum(1, keepdim=True) for h in hs], 1)
    return pooled.cpu().numpy(), fl.cpu().numpy()

XC = extract(wname, whisper_embed, WHISPER_WIN_SEC, WHISPER_BATCH)
del wenc; torch.cuda.empty_cache()
print("C:", XC.shape)

XA_dir = f"{FEAT_DIR}/xlsr_{XLSR_MODEL.split('/')[-1]}_12s"
XA = np.zeros((N, XLSR_KEEP + 1, XB.shape[2]), np.float16)
for s, (idx, _) in GROUPS.items():
    XA[idx] = np.load(f"{XA_dir}/{s}.npy")[:, :XLSR_KEEP + 1]
print("A:", XA.shape)

## 3. Head: weighted-sum + MLP, có thể nhận một hoặc hai mô hình nền

In [ ]:
def to_gpu(X):
    Xg = torch.tensor(X, device=device)
    mtr = torch.tensor(tr, device=device)
    mu, sd = Xg[mtr].float().mean(0), Xg[mtr].float().std(0) + 1e-5
    return Xg, mu, sd

FEATS = {k: to_gpu(v) for k, v in dict(A=XA, B=XB, C=XC).items()}
del XA, XB, XC
SETS = {"A_xlsr12s": ["A"], "B_xlsr_full": ["B"], "C_phowhisper": ["C"], "D_fusion_BC": ["B", "C"]}

class Head(nn.Module):
    def __init__(s, keys):
        super().__init__(); s.keys = keys
        for k in keys:
            _, mu, sd = FEATS[k]
            s.register_buffer(f"mu_{k}", mu); s.register_buffer(f"sd_{k}", sd)
        s.w = nn.ParameterDict({k: nn.Parameter(torch.zeros(FEATS[k][1].shape[0])) for k in keys})
        d = sum(FEATS[k][1].shape[1] for k in keys)
        s.mlp = nn.Sequential(nn.Linear(d, 512), nn.GELU(), nn.Dropout(0.3), nn.Linear(512, C))
    def forward(s, idx):
        parts = []
        for k in s.keys:
            z = (FEATS[k][0][idx].float() - getattr(s, f"mu_{k}")) / getattr(s, f"sd_{k}")
            parts.append((z * s.w[k].softmax(0)[None, :, None]).sum(1))
        return s.mlp(torch.cat(parts, -1))

@torch.no_grad()
def predict(h, mask, bs=2048):
    h.eval(); idx = torch.tensor(np.where(mask)[0], device=device)
    return torch.cat([h(idx[i:i + bs]).softmax(-1) for i in range(0, len(idx), bs)]).cpu().numpy()

def train_head(keys, loss, seed, epochs=80, lr=1e-3, wd=1e-3, bs=256, patience=10):
    torch.manual_seed(seed)
    h = Head(keys).to(device); opt = torch.optim.AdamW(h.parameters(), lr=lr, weight_decay=wd)
    Q = target_matrix(*loss); tr_idx = torch.tensor(np.where(tr)[0], device=device)
    ytr = torch.tensor(y, device=device)
    best, state, bad = -1, None, 0
    for ep in range(epochs):
        h.train(); perm = tr_idx[torch.randperm(len(tr_idx), device=device)]
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            l = -(Q[ytr[b]] * F.log_softmax(h(b), -1)).sum(1).mean()
            opt.zero_grad(); l.backward(); opt.step()
        acc = (predict(h, va).argmax(1) == y[va]).mean()
        if acc > best:
            best, state, bad = acc, {k: v.clone() for k, v in h.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience:
                break
    h.load_state_dict(state)
    return h

RUNS, PV, PT = [], {}, {}
for sname, keys in SETS.items():
    for lname, loss in LOSSES.items():
        t = time.time()
        for s in SEEDS:
            h = train_head(keys, loss, s)
            pv, pt = predict(h, va), predict(h, te)
            PV[(sname, lname, s)], PT[(sname, lname, s)] = pv, pt
            RUNS.append(dict(set=sname, loss=lname, seed=s, **{f"valid_{k}": float(v) for k, v in metrics(pv, y[va]).items()},
                             **{f"test_{k}": float(v) for k, v in metrics(pt, yt).items()},
                             layer_w={k: h.w[k].softmax(0).detach().cpu().numpy().round(4).tolist() for k in keys}))
        r = pd.DataFrame(RUNS[-len(SEEDS):])
        print(f"{sname:14s} {lname:6s} | valid {r.valid_acc.mean():.4f} | test acc {r.test_acc.mean():.4f}±{r.test_acc.std():.4f} "
              f"| F1 {r.test_macro_f1.mean():.4f} | err {r.test_mean_err_km.mean():.0f} km | {time.time() - t:.0f}s", flush=True)
runs = pd.DataFrame(RUNS)

## 4. Ensemble (trung bình xác suất) và chọn mô hình cuối theo valid

In [ ]:
def ens(keys):
    return np.mean([PV[k] for k in keys], 0), np.mean([PT[k] for k in keys], 0)

CANDS = {}
for sname in SETS:
    for lname in LOSSES:
        CANDS[f"{sname}|{lname}|3seed"] = [(sname, lname, s) for s in SEEDS]
for lname in LOSSES:
    CANDS[f"B+C+D|{lname}|ens"] = [(sn, lname, s) for sn in ("B_xlsr_full", "C_phowhisper", "D_fusion_BC") for s in SEEDS]
CANDS["B+C+D|ls+geols|ens"] = [(sn, ln, s) for sn in ("B_xlsr_full", "C_phowhisper", "D_fusion_BC") for ln in LOSSES for s in SEEDS]

ENS = []
for name, keys in CANDS.items():
    pv, pt = ens(keys)
    ENS.append(dict(model=name, **{f"valid_{k}": round(float(v), 4) for k, v in metrics(pv, y[va]).items()},
                    **{f"test_{k}": round(float(v), 4) for k, v in metrics(pt, yt).items()}))
ens_df = pd.DataFrame(ENS).sort_values("valid_acc", ascending=False)
print(ens_df[["model", "valid_acc", "test_acc", "test_macro_f1", "test_region_acc", "test_mean_err_km"]].to_string(index=False))

B = np.random.default_rng(0).integers(0, len(yt), (2000, len(yt)))
def paired(a, b):
    pa_, pb_ = ens(CANDS[a])[1].argmax(1), ens(CANDS[b])[1].argmax(1)
    dacc = (pa_ == yt).astype(float) - (pb_ == yt); dkm = DIST[yt, pa_] - DIST[yt, pb_]
    return dict(acc_diff=round(float(dacc.mean()), 4), acc_ci95=np.percentile(dacc[B].mean(1), [2.5, 97.5]).round(4).tolist(),
                km_diff=round(float(dkm.mean()), 1), km_ci95=np.percentile(dkm[B].mean(1), [2.5, 97.5]).round(1).tolist())
ref = "A_xlsr12s|ls|3seed"
best_ls = ens_df[ens_df.model.str.contains(r"\|ls\||\|ls\+")].iloc[0]["model"]
best_all = ens_df.iloc[0]["model"]
COMPARE = {f"{m} - {ref}": paired(m, ref) for m in dict.fromkeys(
    ["B_xlsr_full|ls|3seed", "C_phowhisper|ls|3seed", "D_fusion_BC|ls|3seed", best_ls, best_all]) if m != ref}
COMPARE[f"{best_all} - D_fusion_BC|ls|3seed"] = paired(best_all, "D_fusion_BC|ls|3seed") if best_all != "D_fusion_BC|ls|3seed" else None
print(f"\nMô hình tốt nhất theo valid: {best_all}")
print(json.dumps(COMPARE, indent=1))

## 5. Probe từng tầng: thông tin phương ngữ nằm ở đâu trong XLS-R và PhoWhisper?

In [ ]:
LAYER_PROBE = {}
if RUN_LAYER_PROBE:
    for k, label in (("B", "XLS-R (toàn câu)"), ("C", "PhoWhisper")):
        Xg, mu, sd = FEATS[k]; accs = []; Y = torch.tensor(y, device=device)
        for l in range(Xg.shape[1]):
            torch.manual_seed(0)
            lin = nn.Linear(Xg.shape[2], C).to(device); opt = torch.optim.AdamW(lin.parameters(), lr=1e-3, weight_decay=1e-3)
            Z = lambda idx: (Xg[idx, l].float() - mu[l]) / sd[l]
            tri = torch.tensor(np.where(tr)[0], device=device); vai = torch.tensor(np.where(va)[0], device=device)
            tei = torch.tensor(np.where(te)[0], device=device); best, bt = -1, 0
            for ep in range(40):
                perm = tri[torch.randperm(len(tri), device=device)]
                for i in range(0, len(perm), 256):
                    b = perm[i:i + 256]; l_ = F.cross_entropy(lin(Z(b)), Y[b])
                    opt.zero_grad(); l_.backward(); opt.step()
                with torch.no_grad():
                    a = (lin(Z(vai)).argmax(1).cpu().numpy() == y[va]).mean()
                    if a > best:
                        best, bt = a, (lin(Z(tei)).argmax(1).cpu().numpy() == yt).mean()
            accs.append(dict(layer=l, valid_acc=round(float(best), 4), test_acc=round(float(bt), 4)))
        LAYER_PROBE[label] = accs
        print(label, [a["valid_acc"] for a in accs])

    import matplotlib.pyplot as plt
    plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False, "font.size": 9,
                         "axes.facecolor": "#fcfcfb", "figure.facecolor": "#fcfcfb", "axes.edgecolor": "#52514e",
                         "xtick.color": "#52514e", "ytick.color": "#52514e"})
    fig, ax = plt.subplots(figsize=(5.6, 3.2))
    n_total = {"XLS-R (toàn câu)": 24, "PhoWhisper": FEATS["C"][0].shape[1] - 1}
    for (label, accs), col in zip(LAYER_PROBE.items(), ("#2a78d6", "#eb6834")):
        xs = [a["layer"] / n_total[label] for a in accs]; ys = [a["valid_acc"] for a in accs]
        ax.plot(xs, ys, color=col, linewidth=2, marker="o", markersize=3, label=label)
        j = int(np.argmax(ys)); ax.annotate(f"tầng {accs[j]['layer']}: {ys[j]:.1%}", (xs[j], ys[j]), fontsize=7,
                                            color="#52514e", xytext=(4, 4), textcoords="offset points")
    ax.set_xlabel("Độ sâu tương đối của tầng (0 = đầu vào, 1 = tầng cuối)"); ax.set_ylabel("Accuracy valid (probe tuyến tính)")
    ax.set_title("Thông tin phương ngữ theo tầng của hai mô hình nền", fontsize=9, loc="left")
    ax.grid(color="#e6e5e0", linewidth=0.6); ax.set_axisbelow(True); ax.legend(frameon=False, loc="best")
    fig.tight_layout(); fig.savefig(f"{RES_DIR}/fig_layers_backbones.png"); plt.show()

In [ ]:
summary = runs.groupby(["set", "loss"])[["valid_acc", "test_acc", "test_macro_f1", "test_top5", "test_region_acc",
                                         "test_mean_err_km"]].agg(["mean", "std"]).round(4)
summary.columns = ["_".join(c) for c in summary.columns]
print(summary.to_string())
report = dict(per_config=summary.reset_index().to_dict(orient="records"), ensembles=ens_df.to_dict(orient="records"),
              compare=COMPARE, best_by_valid=best_all, layer_probe=LAYER_PROBE,
              layer_weights={f"{r['set']}|{r['loss']}|{r['seed']}": r["layer_w"] for r in RUNS},
              losses={k: str(v[1]) for k, v in LOSSES.items()},
              config=dict(xlsr=XLSR_MODEL, xlsr_keep=XLSR_KEEP, xlsr_win=XLSR_WIN_SEC, max_sec=MAX_SEC,
                          whisper=WHISPER_MODEL, whisper_win=WHISPER_WIN_SEC),
              gpu=torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
              total_minutes=round((time.time() - T0) / 60, 1))
json.dump(report, open(f"{RES_DIR}/improve_report.json", "w"), ensure_ascii=False, indent=1, default=str)
print("Đã lưu results/improve_report.json + fig_layers_backbones.png – gửi cho Claude.")